In [1]:
from pathlib import Path
import sys

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

import pickle
from server_config import redcap_path, preprocessed_path

import pandas as pd
import numpy as np
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
import plotly.graph_objects as go


In [2]:


with open(preprocessed_path + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  
    
with open(preprocessed_path + '/ema_meta.pkl', 'rb') as file:
    df_ema_meta = pickle.load(file) 
    
with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")

split_path = redcap_path + "/sp1_cleaned"+"/111025_subject_splits.csv"
df_split = pd.read_csv(split_path)


## 1. Affect 

In [ ]:
df_ema_panas = df_ema_content[df_ema_content["quest_title"].str.startswith("panas_", na=False)]

extra_cols = ["assess", "study", "quest_create", "quest_nr"]

aggregated_info = df_ema_panas.groupby(["customer", "unique_day_id"])[extra_cols].first().reset_index()

# Pivot the table as specified
df_panas_piv = df_ema_panas.pivot_table(
    index=["customer", "unique_day_id", "assess"],
    columns="quest_title",
    values="choice_text",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_panas_piv.columns = [col for col in df_panas_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_panas_piv = df_panas_piv.reset_index()
df_panas_piv = df_panas_piv.drop_duplicates()

In [ ]:
na_cols = ['panas_fear1', 'panas_fear2', 'panas_guilt1','panas_guilt2', 'panas_hostility1', 'panas_hostility2','panas_sadness1', 'panas_sadness2','panas_loneliness']
pa_cols = ['panas_attentiveness', 'panas_joviality1', 'panas_joviality2','panas_selfassurance','panas_serenity1', 'panas_serenity2']

sadness_cols = ['panas_sadness1', 'panas_sadness2']
fear_cols = ['panas_fear1', 'panas_fear2']
hostility_cols = ['panas_hostility1', 'panas_hostility2']
guilt_cols = ['panas_guilt1','panas_guilt2']

joviality_cols = ['panas_joviality1', 'panas_joviality2']
serenity_cols = ['panas_serenity1', 'panas_serenity2']

pa_high_cols = ["panas_attentiveness","panas_joviality1", "panas_joviality2","panas_selfassurance"]
pa_low_cols = ["panas_serenity1", "panas_serenity2"]

na_high_cols = ["panas_fear1", "panas_fear2","panas_hostility1", "panas_hostility2"]
na_low_cols = ["panas_sadness1", "panas_sadness2","panas_loneliness", "panas_fatigue"]


In [ ]:
# all item columns used anywhere
panas_all_cols = sorted(set(
    na_cols
    + pa_cols
    + sadness_cols
    + fear_cols
    + hostility_cols
    + guilt_cols
    + joviality_cols
    + serenity_cols
    + pa_high_cols
    + pa_low_cols
    + na_high_cols
    + na_low_cols
))

# make sure they are numeric
df_panas_piv[panas_all_cols] = df_panas_piv[panas_all_cols].apply(
    pd.to_numeric, errors="coerce"
)


In [ ]:
group_cols = ['customer', 'unique_day_id', 'assess']

scale_defs = {
    'panas_na':        na_cols,
    'panas_pa':        pa_cols,
    'panas_pa_high_ar':pa_high_cols,
    'panas_pa_low_ar': pa_low_cols,
    'panas_na_high_ar':na_high_cols,
    'panas_na_low_ar': na_low_cols,
    'panas_sadness':   sadness_cols,
    'panas_fear':      fear_cols,
    'panas_hostility': hostility_cols,
    'panas_guilt':     guilt_cols,
    'panas_joviality': joviality_cols,
    'panas_serenity':  serenity_cols,
}

for new_col, cols in scale_defs.items():
    df_panas_piv[new_col] = (
        df_panas_piv
        .groupby(group_cols)[cols]
        .transform('mean')
        .mean(axis=1)
    )


### 1.1 Mean and standard deviation

In [ ]:
col_list = ['panas_na','panas_pa','panas_pa_high_ar','panas_pa_low_ar','panas_na_high_ar','panas_na_low_ar','panas_sadness','panas_fear','panas_hostility','panas_guilt','panas_joviality',
            'panas_serenity', 'panas_loneliness', 'panas_fatigue','panas_shyness', 'panas_attentiveness', 'panas_selfassurance']

In [ ]:

df_panas_piv[col_list] = df_panas_piv[col_list].apply(pd.to_numeric, errors='coerce')

agg_affect = (
    df_panas_piv
    .groupby(['customer', 'assess'])[col_list]
    .agg(['mean', 'std'])
)

agg_affect.columns = [
    f"{col}_{stat if stat != 'std' else 'sd'}"
    for col, stat in agg_affect.columns
]

agg_affect = agg_affect.reset_index()

### 1.2 Emotion differentiation

In [ ]:
na_items = ['panas_fear', 'panas_guilt', 'panas_hostility','panas_sadness','panas_loneliness', 'panas_fatigue']
pa_items = ['panas_attentiveness', 'panas_joviality','panas_selfassurance','panas_serenity1']
all_items = na_items + pa_items
df_panas_piv[all_items] = df_panas_piv[all_items].apply(pd.to_numeric, errors='coerce')


In [ ]:
import numpy as np
import pandas as pd

def compute_momentary_ed(df, emotion_cols, id_col="customer", phase_col="assess", prefix="na"):
    """
    Momentary Emotion Differentiation (EDi) nach Erbas et al. (2021),
    getrennt pro Person × Phase und getrennt für ein Set von Emotionen (NA oder PA).

    Neue Spalten im zurückgegebenen df:
      {prefix}_EDi        : momentary ED pro Beep (<= 0; näher 0 = höhere Differentiation)
      {prefix}_ED_overall : Aggregat pro Person × Phase (aus nonED_i, analog zum overall Index)
      {prefix}_n_beeps    : Anzahl gültiger Beeps in dieser Person × Phase
    """
    df = df.copy()

    # Nur die Emotionsspalten verwenden → Gruppierungsspalten sind keine DataFrame-Spalten mehr
    emo = df[emotion_cols].astype(float)

    J = len(emotion_cols)

    def _per_group(g):
        # g: DataFrame mit nur den Emotionsspalten, Index = ursprünglicher df-Index
        valid_rows = ~g.isna().all(axis=1)
        emo_valid = g.loc[valid_rows]
        idx_valid = emo_valid.index
        n = emo_valid.shape[0]

        # Default: alles NaN
        ed = pd.Series(np.nan, index=g.index)
        overall = pd.Series(np.nan, index=g.index)
        n_beeps = pd.Series(n, index=g.index)  # n pro Gruppe, in jede Zeile geschrieben

        if n <= 1:
            return pd.DataFrame({
                f"{prefix}_EDi": ed,
                f"{prefix}_ED_overall": overall,
                f"{prefix}_n_beeps": n_beeps
            }, index=g.index)

        # (1) person-mean zentrieren (innerhalb der Person×Phase-Gruppe)
        centered = emo_valid - emo_valid.mean(axis=0)

        # (4)+(5) Varianzen pro Item und Summe über Items
        denom = centered.var(ddof=1, axis=0).sum()

        if denom <= 0 or np.isnan(denom):
            return pd.DataFrame({
                f"{prefix}_EDi": ed,
                f"{prefix}_ED_overall": overall,
                f"{prefix}_n_beeps": n_beeps
            }, index=g.index)

        # (2) Mittelwert der zentrierten Items pro Beep
        row_mean = centered.mean(axis=1)

        # (3) Zähler: (J * mean)^2
        num = (row_mean * J) ** 2

        # nonED_i & EDi
        noned = num / denom
        ed.loc[idx_valid] = -noned  # näher 0 = höhere Differentiation

        # Aggregierter ED pro Person×Phase
        overall_val = -noned.sum() / (n - 1)
        overall[:] = overall_val

        return pd.DataFrame({
            f"{prefix}_EDi": ed,
            f"{prefix}_ED_overall": overall,
            f"{prefix}_n_beeps": n_beeps
        }, index=g.index)

    # Gruppierung über externe Keys (df[id_col], df[phase_col]) → keine Gruppierungsspalten in g
    res = emo.groupby([df[id_col], df[phase_col]], group_keys=False).apply(_per_group)

    # Ergebnisse über den Index mit df alignen
    df[[f"{prefix}_EDi", f"{prefix}_ED_overall", f"{prefix}_n_beeps"]] = res[
        [f"{prefix}_EDi", f"{prefix}_ED_overall", f"{prefix}_n_beeps"]
    ]

    return df


In [ ]:
# Zuerst NA-ED
df_ed = compute_momentary_ed(
    df_panas_piv,
    emotion_cols=na_items,
    id_col="customer",
    phase_col="assess",
    prefix="na"
)

# Danach PA-ED (baut auf df_ed auf, fügt nur neue Spalten hinzu)
df_ed = compute_momentary_ed(
    df_ed,
    emotion_cols=pa_items,
    id_col="customer",
    phase_col="assess",
    prefix="pa"
)


In [ ]:
def compute_icc_emodiff(df, emotion_cols, id_col="customer", phase_col="assess", prefix="na"):
    """
    Klassischer ICC/Cronbach's-Alpha-basierter Differentiationsindex pro Person × Phase.

    Neue Spalten im zurückgegebenen df:
      {prefix}_ICC_raw      : ICC/Cronbach's alpha (0–1, höher = mehr Kovariation = weniger Differentiation)
      {prefix}_ICC_diff     : 1 - ICC_raw (0–1, höher = mehr Differentiation)
      {prefix}_n_beeps_icc  : Anzahl gültiger Beeps in dieser Person × Phase
    """
    df = df.copy()

    emo = df[emotion_cols].astype(float)
    J = len(emotion_cols)

    def _icc_group(g):
        # g: nur Emotionsspalten, Index = ursprünglicher df-Index
        valid = ~g.isna().all(axis=1)
        emo_valid = g.loc[valid]
        n = emo_valid.shape[0]

        icc_raw = np.nan

        if n > 1 and J > 1:
            item_vars = emo_valid.var(ddof=1)          # Varianz pro Item
            sum_item_vars = item_vars.sum()

            sum_scores = emo_valid.sum(axis=1)         # Sumscore pro Beep
            total_var = sum_scores.var(ddof=1)

            if total_var > 0 and sum_item_vars >= 0:
                icc_raw = (J / (J - 1.0)) * (1.0 - (sum_item_vars / total_var))
                if icc_raw < 0:
                    icc_raw = np.nan

        icc_raw_series = pd.Series(icc_raw, index=g.index)
        icc_diff_series = pd.Series(
            (1.0 - icc_raw) if pd.notna(icc_raw) else np.nan,
            index=g.index
        )
        n_series = pd.Series(n, index=g.index)

        return pd.DataFrame({
            f"{prefix}_ICC_raw": icc_raw_series,
            f"{prefix}_ICC_diff": icc_diff_series,
            f"{prefix}_n_beeps_icc": n_series
        }, index=g.index)

    res = emo.groupby([df[id_col], df[phase_col]], group_keys=False).apply(_icc_group)

    df[[f"{prefix}_ICC_raw", f"{prefix}_ICC_diff", f"{prefix}_n_beeps_icc"]] = res[
        [f"{prefix}_ICC_raw", f"{prefix}_ICC_diff", f"{prefix}_n_beeps_icc"]
    ]

    return df


In [ ]:
# 1) ICC für negative Emotionen pro Person × Phase
df_icc = compute_icc_emodiff(
    df=df_panas_piv,
    emotion_cols=na_items,
    id_col="customer",
    phase_col="assess",
    prefix="na"
)

# 2) ICC für positive Emotionen pro Person × Phase
df_icc = compute_icc_emodiff(
    df=df_icc,
    emotion_cols=pa_items,
    id_col="customer",
    phase_col="assess",
    prefix="pa"
)

In [ ]:
# Phase-Level-ICC (eine Zeile pro customer × assess)
icc_phase = (
    df_icc
    .drop_duplicates(subset=["customer", "assess"])
    [["customer", "assess","unique_day_id",
      "na_ICC_raw", "na_ICC_diff", "na_n_beeps_icc",
      "pa_ICC_raw", "pa_ICC_diff"]]
    .copy()
)


In [ ]:
# Merge PANAS-EMA mit momentary und aggregierten ED-Maßen
df_merged = agg_affect.merge(
    df_ed[
        ["customer", "assess","unique_day_id",
         "na_EDi", "na_ED_overall",
         "pa_EDi", "pa_ED_overall"]
    ],
    on=["customer", "assess"],
    how="left"
)

df_affect_final = df_merged.merge(
    icc_phase,
    on=["customer", "assess", "unique_day_id"],
    how="left"
)


In [ ]:
df_affect_final.customer.nunique()

In [ ]:
df_monitoring_final = df_affect_final.merge(df_monitoring, on = "customer", how= "left")

In [ ]:
df_monitoring_final.groupby("study_version")["customer"].nunique()

In [ ]:
import numpy as np
from scipy.stats import ttest_rel
import plotly.graph_objects as go

# -------------------------------------------------------
# 0) Filter: nur Personen mit >10 Beeps in Phase 0 & 1
# -------------------------------------------------------
eligible_customers = (
    df_affect_final.query("assess in [0,1]")
    .drop_duplicates(["customer", "assess"])
    .pivot(index="customer", columns="assess", values="na_n_beeps_icc")
    .pipe(lambda d: d[(d[0] > 10) & (d[1] > 10)].index)
)

df_affect_final_filtered = df_affect_final[
    df_affect_final["customer"].isin(eligible_customers)
].copy()

# -------------------------------------------------------
# 1) Phase-level data, nur Phasen 0 und 1
# -------------------------------------------------------
phase_level = (
    df_affect_final_filtered
    .drop_duplicates(subset=["customer", "assess"])
    .copy()
)

phase_level = phase_level[phase_level["assess"].isin([0, 1])].copy()
phase_level["assess"] = phase_level["assess"].astype(int)

# -------------------------------------------------------
# 2) Label-Map ohne "(mean)" und mit ICC-Labels
# -------------------------------------------------------
label_map = {
    "panas_na_mean": "Negative affect",
    "panas_na_sd": "Negative affect (SD)",

    "panas_pa_mean": "Positive affect",
    "panas_pa_sd": "Positive affect (SD)",

    "panas_pa_high_ar_mean": "High-arousal PA",
    "panas_pa_high_ar_sd": "High-arousal PA (SD)",

    "panas_pa_low_ar_mean": "Low-arousal PA",
    "panas_pa_low_ar_sd": "Low-arousal PA (SD)",

    "panas_na_high_ar_mean": "High-arousal NA",
    "panas_na_high_ar_sd": "High-arousal NA (SD)",

    "panas_na_low_ar_mean": "Low-arousal NA",
    "panas_na_low_ar_sd": "Low-arousal NA (SD)",

    "panas_sadness_mean": "Sadness",
    "panas_sadness_sd": "Sadness (SD)",

    "panas_fear_mean": "Fear",
    "panas_fear_sd": "Fear (SD)",

    "panas_hostility_mean": "Hostility",
    "panas_hostility_sd": "Hostility (SD)",

    "panas_guilt_mean": "Guilt",
    "panas_guilt_sd": "Guilt (SD)",

    "panas_joviality_mean": "Joviality",
    "panas_joviality_sd": "Joviality (SD)",

    "panas_serenity_mean": "Serenity",
    "panas_serenity_sd": "Serenity (SD)",

    "panas_loneliness_mean": "Loneliness",
    "panas_loneliness_sd": "Loneliness (SD)",

    "panas_fatigue_mean": "Fatigue",
    "panas_fatigue_sd": "Fatigue (SD)",

    "panas_shyness_mean": "Shyness",
    "panas_shyness_sd": "Shyness (SD)",

    "panas_attentiveness_mean": "Attentiveness",
    "panas_attentiveness_sd": "Attentiveness (SD)",

    "panas_selfassurance_mean": "Self-Assurance",
    "panas_selfassurance_sd": "Self-Assurance (SD)",

    "na_ICC_diff": "NA differentiation (ICC)",
    "pa_ICC_diff": "PA differentiation (ICC)",
}

# -------------------------------------------------------
# 3) Gewünschte Reihenfolge der Variablen (per Code-Name)
# -------------------------------------------------------
desired_order = [
    # Einzelfacetten NA
    "panas_fatigue_mean",
    "panas_fear_mean",
    "panas_guilt_mean",
    "panas_loneliness_mean",
    "panas_sadness_mean",
    "panas_shyness_mean",
    "panas_hostility_mean",
    # Einzelfacetten PA
    "panas_joviality_mean",
    "panas_serenity_mean",
    "panas_attentiveness_mean",
    "panas_selfassurance_mean",
    # Arousal-spezifische NA/PA
    "panas_na_high_ar_mean",
    "panas_na_low_ar_mean",
    "panas_pa_high_ar_mean",
    "panas_pa_low_ar_mean",
    # Gesamt-NA/PA
    "panas_na_mean",
    "panas_pa_mean",
    # Differentiation
    "na_ICC_diff",
    "pa_ICC_diff",
]

# tatsächlich vorhandene Variablen (falls z.B. Attentiveness noch fehlt)
vars_to_plot = [v for v in desired_order if v in phase_level.columns]

# -------------------------------------------------------
# 4) Valenz-Sets für Farbcodierung
# -------------------------------------------------------
positive_vars = {
    "panas_joviality_mean",
    "panas_serenity_mean",
    "panas_attentiveness_mean",
    "panas_selfassurance_mean",
    "panas_pa_high_ar_mean",
    "panas_pa_low_ar_mean",
    "panas_pa_mean",
    "pa_ICC_diff",
}

negative_vars = {
    "panas_fatigue_mean",
    "panas_fear_mean",
    "panas_guilt_mean",
    "panas_loneliness_mean",
    "panas_sadness_mean",
    "panas_shyness_mean",
    "panas_hostility_mean",
    "panas_na_high_ar_mean",
    "panas_na_low_ar_mean",
    "panas_na_mean",
    "na_ICC_diff",
}

# -------------------------------------------------------
# 5) Stats pro Variable (gepaarte T-Tests 0 vs 1)
# -------------------------------------------------------
results = []

for var in vars_to_plot:
    df_var = phase_level[["customer", "assess", var]].dropna()

    wide = df_var.pivot(index="customer", columns="assess", values=var)

    if (0 not in wide.columns) or (1 not in wide.columns):
        continue

    wide = wide.dropna(subset=[0, 1])
    if wide.shape[0] < 2:
        continue

    x0 = wide[0]
    x1 = wide[1]
    diff = x1 - x0

    mean0 = x0.mean()
    mean1 = x1.mean()
    delta = diff.mean()
    sd_diff = diff.std(ddof=1)
    se_diff = sd_diff / np.sqrt(len(diff))
    ci95 = 1.96 * se_diff

    if sd_diff == 0:
        t, p = 0.0, 1.0
    else:
        t, p = ttest_rel(x0, x1)

    results.append({
        "variable": var,
        "n": len(diff),
        "mean_phase0": mean0,
        "mean_phase1": mean1,
        "delta": delta,      # phase1 - phase0
        "sd_diff": sd_diff,
        "ci95": ci95,
        "p_raw": p,
    })

results_df = pd.DataFrame(results)
if results_df.empty:
    raise ValueError("No variables had enough complete pairs for 0 vs 1.")

# -------------------------------------------------------
# 6) Bonferroni-Korrektur + Sortierung in Wunschreihenfolge
# -------------------------------------------------------
m = results_df.shape[0]
results_df["p_bonf"] = np.minimum(results_df["p_raw"] * m, 1.0)
results_df["significant"] = results_df["p_bonf"] < 0.05

order_map = {var: i for i, var in enumerate(desired_order)}
results_df["order"] = results_df["variable"].map(order_map)
results_df = results_df.sort_values("order").reset_index(drop=True)

results_df["sig_star"] = np.where(results_df["p_bonf"] < 0.05, " *", "")

# lesbare Labels (ohne "mean", mit ICC)
results_df["nice_label"] = results_df["variable"].map(label_map).fillna(results_df["variable"])
y_labels = results_df["nice_label"] + results_df["sig_star"]

# -------------------------------------------------------
# 7) Farb-Logik: 
#    - nicht signifikant: grau
#    - signifikant & positive Skala: blau
#    - signifikant & negative Skala: rot
# -------------------------------------------------------
colors = []
for _, row in results_df.iterrows():
    var = row["variable"]
    if not row["significant"]:
        colors.append("gray")
    else:
        if var in positive_vars:
            colors.append("blue")
        elif var in negative_vars:
            colors.append("crimson")
        else:
            colors.append("black")  # Fallback, falls etwas nicht klassifiziert ist

# -------------------------------------------------------
# 8) Plot (vertikal, eine Achse, CI-Balken)
# -------------------------------------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=results_df["delta"],          # Effektgröße (Phase1 - Phase0)
        y=y_labels,                     # Variablen in gewünschter Reihenfolge
        mode="markers",
        error_x=dict(
            type="data",
            array=results_df["ci95"],   # 95%-CI der Differenz
            visible=True
        ),
        marker=dict(
            size=8,
            color=colors
        ),
        text=[
            f"mean0={row.mean_phase0:.2f}<br>"
            f"mean1={row.mean_phase1:.2f}<br>"
            f"delta={row.delta:.2f}<br>"
            f"p={row.p_raw:.3g}, p_bonf={row.p_bonf:.3g}"
            for _, row in results_df.iterrows()
        ],
        hovertemplate="%{y}<br>%{text}<extra></extra>",
    )
)

# vertikale 0-Linie = keine Veränderung
fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title="Change from phase 0 to phase 1 (phase1 − phase0)",
    xaxis_title="Mean difference",
    yaxis_title="Variable",
    height=max(700, 30 * len(results_df)),
    width=900,
    showlegend=False,
)

# erste Variable oben
fig.update_yaxes(autorange="reversed")

fig.show()


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr, pearsonr

# optional (if Plotly figures sometimes don't show in your environment):
# import plotly.io as pio
# pio.renderers.default = "plotly_mimetype"
# pio.renderers.default = "notebook_connected"

def _nice(v):
    return label_map.get(v, v)

def corr_heatmap_baseline_sig(df_wide, title, var_order, method="spearman", alpha=0.05):
    """
    df_wide: index=customer, columns=variables (wide, one row per person)
    var_order: list of variables in desired order
    """
    # keep only vars that exist and have variance / enough data
    keep = []
    for v in var_order:
        if v not in df_wide.columns:
            continue
        x = pd.to_numeric(df_wide[v], errors="coerce")
        if x.notna().sum() < 3:
            continue
        if x.std(ddof=1) == 0:
            continue
        keep.append(v)

    df_use = df_wide[keep].apply(pd.to_numeric, errors="coerce")
    k = len(keep)
    labels = [_nice(v) for v in keep]

    # pairwise r, p, n
    R = np.full((k, k), np.nan, dtype=float)
    P = np.full((k, k), np.nan, dtype=float)
    N = np.zeros((k, k), dtype=int)

    tested_pairs = []  # list of (i,j) where a p-value was computed

    for i in range(k):
        R[i, i] = 1.0
        P[i, i] = 0.0
        N[i, i] = int(df_use.iloc[:, i].notna().sum())

    for i in range(k):
        xi = df_use.iloc[:, i]
        for j in range(i + 1, k):
            xj = df_use.iloc[:, j]
            pair = pd.concat([xi, xj], axis=1).dropna()
            n = pair.shape[0]
            N[i, j] = N[j, i] = n

            if n < 3:
                continue
            # constant within pair -> undefined correlation
            if pair.iloc[:, 0].std(ddof=1) == 0 or pair.iloc[:, 1].std(ddof=1) == 0:
                continue

            if method == "spearman":
                r, p = spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
                cbar_title = "Spearman ρ"
            elif method == "pearson":
                r, p = pearsonr(pair.iloc[:, 0], pair.iloc[:, 1])
                cbar_title = "Pearson r"
            else:
                raise ValueError("method must be 'spearman' or 'pearson'")

            R[i, j] = R[j, i] = r
            P[i, j] = P[j, i] = p
            tested_pairs.append((i, j))

    # Bonferroni correction across *tested* off-diagonal pairs
    m = len(tested_pairs)
    P_bonf = np.full((k, k), np.nan, dtype=float)
    if m > 0:
        for (i, j) in tested_pairs:
            pb = min(P[i, j] * m, 1.0)
            P_bonf[i, j] = P_bonf[j, i] = pb
        np.fill_diagonal(P_bonf, 0.0)

    sig = (P_bonf < alpha)

    # text: r + star for Bonferroni-significant (off-diagonal)
    text = np.empty((k, k), dtype=object)
    for i in range(k):
        for j in range(k):
            if np.isfinite(R[i, j]):
                star = "*" if (i != j and sig[i, j]) else ""
                text[i, j] = f"{R[i, j]:.2f}{star}"
            else:
                text[i, j] = ""

    # hover text with p-values and n
    hover = np.empty((k, k), dtype=object)
    for i in range(k):
        for j in range(k):
            if i == j:
                hover[i, j] = f"{labels[i]}<br>n={N[i,j]}"
            else:
                r_str = f"{R[i,j]:.3f}" if np.isfinite(R[i,j]) else "NA"
                p_str = f"{P[i,j]:.3g}" if np.isfinite(P[i,j]) else "NA"
                pb_str = f"{P_bonf[i,j]:.3g}" if np.isfinite(P_bonf[i,j]) else "NA"
                hover[i, j] = (
                    f"{labels[i]} vs {labels[j]}"
                    f"<br>r={r_str}"
                    f"<br>n_pair={N[i,j]}"
                    f"<br>p_raw={p_str}"
                    f"<br>p_bonf={pb_str}"
                    f"<br>Bonferroni m={m}"
                )

    fig = go.Figure(
        data=go.Heatmap(
            z=R,
            x=labels,
            y=labels,
            zmin=-1, zmax=1, zmid=0,
            colorscale="RdBu",
            colorbar=dict(title=cbar_title),
            text=text,
            texttemplate="%{text}",
            hovertext=hover,
            hovertemplate="%{hovertext}<extra></extra>",
            hoverongaps=False,
        )
    )

    fig.update_layout(
        title=f"{title} (method={method}, Bonferroni α={alpha}, m={m}; '*' = significant)",
        width=980,
        height=900,
        xaxis=dict(tickangle=45),
        yaxis=dict(autorange="reversed"),
    )
    fig.show()

    # return matrices if you want to inspect/export them
    corr_df = pd.DataFrame(R, index=keep, columns=keep)
    p_raw_df = pd.DataFrame(P, index=keep, columns=keep)
    p_bonf_df = pd.DataFrame(P_bonf, index=keep, columns=keep)
    n_df = pd.DataFrame(N, index=keep, columns=keep)
    return corr_df, p_raw_df, p_bonf_df, n_df


# =========================================================
# Baseline only (assess = 0), one row per person
# =========================================================
phase_level_unique = phase_level.drop_duplicates(["customer", "assess"]).copy()

baseline0 = (
    phase_level_unique.loc[phase_level_unique["assess"] == 0, ["customer"] + vars_to_plot]
    .set_index("customer")
)

corr0, p_raw0, p_bonf0, n0 = corr_heatmap_baseline_sig(
    baseline0,
    title="Baseline correlations across participants (assess = 0)",
    var_order=vars_to_plot,
    method="spearman",
    alpha=0.05
)


In [ ]:
import plotly.figure_factory as ff

# ---------------------------------------------------
# 0) Label map mit klaren Namen
# ---------------------------------------------------
label_map = {
    "panas_na_mean": "Negative affect",
    "panas_na_sd": "Negative affect (SD)",

    "panas_pa_mean": "Positive affect",
    "panas_pa_sd": "Positive affect (SD)",

    "panas_pa_high_ar_mean": "High-arousal PA",
    "panas_pa_high_ar_sd": "High-arousal PA (SD)",

    "panas_pa_low_ar_mean": "Low-arousal PA",
    "panas_pa_low_ar_sd": "Low-arousal PA (SD)",

    "panas_na_high_ar_mean": "High-arousal NA",
    "panas_na_high_ar_sd": "High-arousal NA (SD)",

    "panas_na_low_ar_mean": "Low-arousal NA",
    "panas_na_low_ar_sd": "Low-arousal NA (SD)",

    "panas_sadness_mean": "Sadness",
    "panas_sadness_sd": "Sadness (SD)",

    "panas_fear_mean": "Fear",
    "panas_fear_sd": "Fear (SD)",

    "panas_hostility_mean": "Hostility",
    "panas_hostility_sd": "Hostility (SD)",

    "panas_guilt_mean": "Guilt",
    "panas_guilt_sd": "Guilt (SD)",

    "panas_joviality_mean": "Joviality",
    "panas_joviality_sd": "Joviality (SD)",

    "panas_serenity_mean": "Serenity",
    "panas_serenity_sd": "Serenity (SD)",

    "panas_loneliness_mean": "Loneliness",
    "panas_loneliness_sd": "Loneliness (SD)",

    "panas_fatigue_mean": "Fatigue",
    "panas_fatigue_sd": "Fatigue (SD)",

    "panas_shyness_mean": "Shyness",
    "panas_shyness_sd": "Shyness (SD)",

    "panas_attentiveness_mean": "Attentiveness",
    "panas_attentiveness_sd": "Attentiveness (SD)",

    "panas_selfassurance_mean": "Self-Assurance",
    "panas_selfassurance_sd": "Self-Assurance (SD)",

    "na_ICC_diff": "NA differentiation (ICC)",
    "pa_ICC_diff": "PA differentiation (ICC)",
}

# ---------------------------------------------------
# 1) Valenz-Sets: welche Variablen gelten als positiv/negativ?
# ---------------------------------------------------
positive_vars = {
    "panas_pa_mean",
    "panas_pa_high_ar_mean",
    "panas_pa_low_ar_mean",
    "panas_joviality_mean",
    "panas_serenity_mean",
    "panas_selfassurance_mean",
    "panas_attentiveness_mean",
    "pa_ICC_diff",
}

negative_vars = {
    "panas_na_mean",
    "panas_na_high_ar_mean",
    "panas_na_low_ar_mean",
    "panas_sadness_mean",
    "panas_fear_mean",
    "panas_guilt_mean",
    "panas_loneliness_mean",
    "panas_fatigue_mean",
    "panas_shyness_mean",
    "panas_hostility_mean",
    "na_ICC_diff",
}

# warme Palette für negative Variablen
neg_palette = [
    "#e41a1c",  # rot
    "#ff7f00",  # orange
    "#b2182b",  # dunkles rot
    "#fb9a99",  # rosa
    "#f781bf",  # pink
    "#a65628",  # braun
    "#d95f02",  # orangebraun
    "#e31a1c",  # rot-Variante
    "#f16913",  # orange-rot
    "#bc80bd",  # violett
    "#fdae61",  # hellorange
    "#fd8d3c",  # orange
]

# kühle Palette für positive Variablen
pos_palette = [
    "#1f78b4",  # blau
    "#33a02c",  # grün
    "#6a3d9a",  # lila
    "#b2df8a",  # hellgrün
    "#a6cee3",  # hellblau
    "#66c2a5",  # türkis
    "#01665e",  # dunkles türkis
    "#41ab5d",  # mittelgrün
]

neutral_palette = ["#636363", "#969696"]  # falls etwas weder klar positiv noch negativ ist

# ---------------------------------------------------
# 2) Baseline data (assess = 0)
# ---------------------------------------------------
baseline_affect = phase_level[phase_level["assess"] == 0].copy()

# ---------------------------------------------------
# 3) Helper to build a distplot from raw values
# ---------------------------------------------------
def make_distplot(df, var_list, title, xlab="Raw value"):
    hist_data = []
    group_labels = []
    colors = []

    pos_idx = 0
    neg_idx = 0
    neu_idx = 0

    for v in var_list:
        if v not in df.columns:
            continue

        x = df[v].dropna()
        if x.empty:
            continue

        hist_data.append(x.values)
        # hier kommen die klaren Namen rein
        group_labels.append(label_map.get(v, v))

        if v in positive_vars:
            colors.append(pos_palette[pos_idx % len(pos_palette)])
            pos_idx += 1
        elif v in negative_vars:
            colors.append(neg_palette[neg_idx % len(neg_palette)])
            neg_idx += 1
        else:
            colors.append(neutral_palette[neu_idx % len(neutral_palette)])
            neu_idx += 1

    if not hist_data:
        print("No non-empty variables for plotting.")
        return

    fig = ff.create_distplot(
        hist_data,
        group_labels,
        show_hist=False,
        show_rug=False,
        colors=colors,
    )

    fig.update_layout(
        title=title,
        xaxis_title=xlab,
        yaxis_title="Density",
        width=900,
        height=500,
    )
    fig.show()

# ---------------------------------------------------
# 4) Plot 1: spezifische Facetten (ohne ICC)
# ---------------------------------------------------
vars_affect_facets = [
    "panas_sadness_mean",
    "panas_fear_mean",
    "panas_guilt_mean",
    "panas_joviality_mean",
    "panas_selfassurance_mean",
    "panas_attentiveness_mean",
    "panas_serenity_mean",
    "panas_loneliness_mean",
    "panas_fatigue_mean",
    "panas_shyness_mean",
    "panas_hostility_mean",
]

make_distplot(
    baseline_affect,
    vars_affect_facets,
    title="Baseline distributions of affect facets (raw values, assess = 0)",
)

# ---------------------------------------------------
# 5) Plot 2: NA + PA differentiation (ICC-basiert)
# ---------------------------------------------------
vars_icc = ["na_ICC_diff", "pa_ICC_diff"]

make_distplot(
    baseline_affect,
    vars_icc,
    title="Baseline distributions of NA/PA differentiation (ICC-based, assess = 0)",
)

# ---------------------------------------------------
# 6) Plot 3: Aggregierte Werte (NA, PA, high/low arousal)
# ---------------------------------------------------
vars_aggregated = [
    "panas_na_mean",          # Gesamt NA
    "panas_pa_mean",          # Gesamt PA
    "panas_na_high_ar_mean",  # high-arousal NA
    "panas_pa_high_ar_mean",  # high-arousal PA
    "panas_na_low_ar_mean",   # low-arousal NA
    "panas_pa_low_ar_mean",   # low-arousal PA
]

make_distplot(
    baseline_affect,
    vars_aggregated,
    title="Baseline distributions of aggregated affect (raw values, assess = 0)",
)


### 2. Emotion Regulation

In [ ]:
df_ema_er = df_ema_content[df_ema_content["quest_title"].str.startswith("er_", na=False)]

extra_cols = ["assess", "study", "quest_create", "quest_nr"]

aggregated_info = df_ema_er.groupby(["customer", "unique_day_id"])[extra_cols].first().reset_index()

# Pivot the table as specified
df_er_piv = df_ema_er.pivot_table(
    index=["customer", "unique_day_id", "assess"],
    columns="quest_title",
    values="choice_text",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_er_piv.columns = [col for col in df_er_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_er_piv = df_er_piv.reset_index()
df_er_piv = df_er_piv.drop_duplicates()

In [ ]:
col_list_er = ['er_acceptance','er_control', 'er_distraction','er_intensity','er_reappraisal','er_relaxation','er_rumination','er_suppression']

In [ ]:

df_er_piv[col_list_er] = df_er_piv[col_list_er].apply(pd.to_numeric, errors='coerce')

agg_er = (
    df_er_piv
    .groupby(['customer', 'assess'])[col_list_er]
    .agg(['mean', 'std'])
)

agg_er.columns = [
    f"{col}_{stat if stat != 'std' else 'sd'}"
    for col, stat in agg_er.columns
]

agg_er = agg_er.reset_index()

In [ ]:

# ------------------------------------------------------------------
# 2) Moment-level indices (per beep)
#    - between-strategy SD at each beep
#    - mean ER-endorsement
#    - repertoire size (optional)
# ------------------------------------------------------------------

# SD across strategies within a beep = between-strategy variability (momentary)
df_er_piv["er_between_sd_beep"] = df_er_piv[col_list_er].std(axis=1, ddof=1)

# mean endorsement within a beep
df_er_piv["er_mean_endorse_beep"] = df_er_piv[col_list_er].mean(axis=1)

# number of strategies "used" at this beep (e.g. > 0)
df_er_piv["er_n_strategies_beep"] = (df_er_piv[col_list_er] > 1).sum(axis=1)

# ------------------------------------------------------------------
# 3) Person × phase indices
#    - mean between-strategy SD over beeps
#    - within-strategy SD over time, averaged across strategies
#    - mean overall ER endorsement
#    - number of beeps
# ------------------------------------------------------------------

# 3a) Between-strategy variability: mean SD_across_strategies over beeps
er_between_phase = (
    df_er_piv
    .groupby(["customer", "assess"], as_index=False)["er_between_sd_beep"]
    .mean()
    .rename(columns={"er_between_sd_beep": "er_between_sd_mean"})
)

# 3b) Within-strategy variability:
#     SD over beeps for each strategy, then mean across strategies
er_within_phase = (
    df_er_piv
    .groupby(["customer", "assess"])[col_list_er]
    .std(ddof=1)           # SD over time per strategy
    .reset_index()
)

er_within_phase["er_within_sd_mean"] = er_within_phase[col_list_er].mean(axis=1)

# 3c) Mean overall ER endorsement per person×phase
er_mean_phase = (
    df_er_piv
    .groupby(["customer", "assess"])[col_list_er]
    .mean()
    .reset_index()
)

er_mean_phase["er_mean_overall"] = er_mean_phase[col_list_er].mean(axis=1)

# 3d) Number of beeps per person×phase
n_beeps_phase = (
    df_er_piv
    .groupby(["customer", "assess"], as_index=False)
    .size()
    .rename(columns={"size": "er_n_beeps"})
)

er_rep_phase = (
    df_er_piv
    .groupby(["customer", "assess"], as_index=False)["er_n_strategies_beep"]
    .mean()
    .rename(columns={"er_n_strategies_beep": "er_n_strategies_mean"})
)

er_flex_phase = (
    er_between_phase
    .merge(er_within_phase[["customer", "assess", "er_within_sd_mean"]],
           on=["customer", "assess"], how="left")
    .merge(er_mean_phase[["customer", "assess", "er_mean_overall"]],
           on=["customer", "assess"], how="left")
    .merge(n_beeps_phase, on=["customer", "assess"], how="left")
    .merge(er_rep_phase, on=["customer", "assess"], how="left")
)


# optional: require a minimum number of beeps per phase (e.g. ≥10)
#er_flex_phase = er_flex_phase.query("er_n_beeps >= 10").reset_index(drop=True)


In [ ]:
df_er_final = er_flex_phase.merge(agg_er, on= ["customer", "assess"])

In [ ]:
import numpy as np
from scipy.stats import ttest_rel
import plotly.graph_objects as go

# -------------------------------------------------------
# 0) Filter: nur Personen mit >10 Beeps in Phase 0 & 1
# -------------------------------------------------------
eligible_customers = (
    df_er_final.query("assess in [0,1]")
    .drop_duplicates(["customer", "assess"])
    .pivot(index="customer", columns="assess", values="er_n_beeps")
    .pipe(lambda d: d[(d[0] > 10) & (d[1] > 10)].index)
)

df_er_final_filtered = df_er_final[df_er_final["customer"].isin(eligible_customers)].copy()

# -------------------------------------------------------
# 1) Phase-level data, phases 0 and 1
# -------------------------------------------------------
phase_level_er = (
    df_er_final_filtered
    .drop_duplicates(subset=["customer", "assess"])
    .copy()
)

phase_level_er = phase_level_er[phase_level_er["assess"].isin([0, 1])].copy()
phase_level_er["assess"] = phase_level_er["assess"].astype(int)

# -------------------------------------------------------
# 2) Variablen + Labels
# -------------------------------------------------------
all_vars = [
    "er_between_sd_mean", 
    "er_within_sd_mean",
    "er_n_strategies_mean",
    "er_mean_overall",
    "er_acceptance_mean",
    "er_acceptance_sd", 
    "er_control_mean", 
    "er_control_sd",
    "er_distraction_mean", 
    "er_distraction_sd", 
    "er_intensity_mean",
    "er_intensity_sd", 
    "er_reappraisal_mean", 
    "er_reappraisal_sd",
    "er_relaxation_mean", 
    "er_relaxation_sd", 
    "er_rumination_mean",
    "er_rumination_sd", 
    "er_suppression_mean", 
    "er_suppression_sd"
]

label_map = {
    "er_between_sd_mean": "Between-Strategy Variability",
    "er_within_sd_mean": "Within-Strategy Variability",
    "er_n_strategies_mean": "Number of strategies used",
    "er_mean_overall": "Overall ER Intensity",
    "er_acceptance_mean": "Acceptance",
    "er_control_mean": "Situation Controlability",
    "er_distraction_mean": "Distraction",
    "er_intensity_mean": "Negative Affect Intensity",
    "er_reappraisal_mean": "Reappraisal",
    "er_relaxation_mean": "Relaxation",
    "er_rumination_mean": "Rumination",
    "er_suppression_mean": "Suppression",
    # optional: SD-Labels
    "er_acceptance_sd": "Acceptance (SD)",
    "er_control_sd": "Control (SD)",
    "er_distraction_sd": "Distraction (SD)",
    "er_intensity_sd": "Intensity (SD)",
    "er_reappraisal_sd": "Reappraisal (SD)",
    "er_relaxation_sd": "Relaxation (SD)",
    "er_rumination_sd": "Rumination (SD)",
    "er_suppression_sd": "Suppression (SD)",
}

# gewünschte Reihenfolge
desired_order = [
    "er_acceptance_mean",
    "er_reappraisal_mean",
    "er_relaxation_mean",
    "er_rumination_mean",
    "er_suppression_mean",
    "er_distraction_mean",
    "er_between_sd_mean",
    "er_within_sd_mean",
    "er_mean_overall",
    "er_intensity_mean",
    "er_control_mean",
]

# nur diese Variablen wirklich plotten (Schnittmenge mit vorhandenen Spalten)
vars_to_plot = [v for v in desired_order if v in phase_level_er.columns]

# Sets für Farbcodierung
adaptive_vars = {
    "er_acceptance_mean",
    "er_reappraisal_mean",
    "er_relaxation_mean",
    "er_distraction_mean",
    "er_control_mean",
    "er_within_sd_mean",


}

maladaptive_vars = {
    "er_rumination_mean",
    "er_suppression_mean",
    "er_intensity_mean",
}

metric_vars = {

}

# -------------------------------------------------------
# 3) Stats per variable (paired 0 vs 1)
# -------------------------------------------------------
results = []

for var in vars_to_plot:
    df_var = phase_level_er[["customer", "assess", var]].dropna()

    wide = df_var.pivot(index="customer", columns="assess", values=var)

    if (0 not in wide.columns) or (1 not in wide.columns):
        continue

    wide = wide.dropna(subset=[0, 1])
    if wide.shape[0] < 2:
        continue

    x0 = wide[0]
    x1 = wide[1]
    diff = x1 - x0

    mean0 = x0.mean()
    mean1 = x1.mean()
    delta = diff.mean()
    sd_diff = diff.std(ddof=1)
    se_diff = sd_diff / np.sqrt(len(diff))
    ci95 = 1.96 * se_diff

    if sd_diff == 0:
        t, p = 0.0, 1.0
    else:
        t, p = ttest_rel(x0, x1)

    results.append({
        "variable": var,
        "n": len(diff),
        "mean_phase0": mean0,
        "mean_phase1": mean1,
        "delta": delta,      # phase1 - phase0
        "sd_diff": sd_diff,
        "ci95": ci95,
        "p_raw": p,
    })

results_df = pd.DataFrame(results)
if results_df.empty:
    raise ValueError("No variables had enough complete pairs for 0 vs 1.")

# -------------------------------------------------------
# 4) Bonferroni correction
# -------------------------------------------------------
m = results_df.shape[0]
results_df["p_bonf"] = np.minimum(results_df["p_raw"] * m, 1.0)
results_df["significant"] = results_df["p_bonf"] < 0.05

# Sortierung in gewünschter Reihenfolge
order_map = {v: i for i, v in enumerate(desired_order)}
results_df["order"] = results_df["variable"].map(order_map)
results_df = results_df.sort_values("order").reset_index(drop=True)

results_df["sig_star"] = np.where(results_df["p_bonf"] < 0.05, " *", "")

# lesbare Labels
results_df["nice_label"] = results_df["variable"].map(label_map).fillna(results_df["variable"])
y_labels = results_df["nice_label"] + results_df["sig_star"]

# -------------------------------------------------------
# 5) Farb-Logik:
#    - nicht signifikant: grau
#    - signifikant & adaptive Strategie: blau
#    - signifikant & maladaptive Strategie: rot
#    - signifikant & Kennwerte (Variabilität, Intensität, Control): grün
# -------------------------------------------------------
colors = []
for _, row in results_df.iterrows():
    var = row["variable"]
    if not row["significant"]:
        colors.append("gray")
    else:
        if var in adaptive_vars:
            colors.append("blue")
        elif var in maladaptive_vars:
            colors.append("crimson")
        elif var in metric_vars:
            colors.append("green")
        else:
            colors.append("black")  # Fallback (sollte hier praktisch nicht vorkommen)

# -------------------------------------------------------
# 6) Plot
# -------------------------------------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=results_df["delta"],          # Mean difference (Phase1 - Phase0)
        y=y_labels,                     # sortiert in gewünschter Reihenfolge
        mode="markers",
        error_x=dict(
            type="data",
            array=results_df["ci95"],   # 95%-CI
            visible=True
        ),
        marker=dict(
            size=8,
            color=colors
        ),
        text=[
            f"mean0={row.mean_phase0:.2f}<br>"
            f"mean1={row.mean_phase1:.2f}<br>"
            f"delta={row.delta:.2f}<br>"
            f"p={row.p_raw:.3g}, p_bonf={row.p_bonf:.3g}"
            for _, row in results_df.iterrows()
        ],
        hovertemplate="%{y}<br>%{text}<extra></extra>",
    )
)

fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title="Change in ER features from phase 0 to phase 1 (phase1 − phase0)",
    xaxis_title="Mean difference",
    yaxis_title="",
    height=max(600, 30 * len(results_df)),
    width=800,
    showlegend=False,
)

fig.update_yaxes(autorange="reversed")

fig.show()


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr, pearsonr

# -------------------------------------------------------
# Helper: stars based on Bonferroni-corrected p
# -------------------------------------------------------
def p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

# -------------------------------------------------------
# 3) Pairwise correlations + p-values (Baseline)
# -------------------------------------------------------
method = "spearman"  # or "pearson"

cols = list(X.columns)
k = len(cols)

rho = np.full((k, k), np.nan, dtype=float)
p_raw = np.full((k, k), np.nan, dtype=float)
n_pair = np.zeros((k, k), dtype=int)

for i in range(k):
    rho[i, i] = 1.0
    p_raw[i, i] = np.nan
    n_pair[i, i] = X[cols[i]].notna().sum()

for i in range(k):
    xi = X[cols[i]]
    for j in range(i + 1, k):
        xj = X[cols[j]]
        pair = pd.concat([xi, xj], axis=1).dropna()
        n = pair.shape[0]
        n_pair[i, j] = n_pair[j, i] = n

        if n < 3:
            continue

        if method == "spearman":
            r, p = spearmanr(pair.iloc[:, 0].values, pair.iloc[:, 1].values)
        else:
            r, p = pearsonr(pair.iloc[:, 0].values, pair.iloc[:, 1].values)

        rho[i, j] = rho[j, i] = r
        p_raw[i, j] = p_raw[j, i] = p

# number of tests for Bonferroni = number of valid unique pairs
upper = np.triu_indices(k, 1)
valid = np.isfinite(p_raw[upper])
m_tests = int(valid.sum()) if valid.any() else 0

p_bonf = np.full((k, k), np.nan, dtype=float)
if m_tests > 0:
    p_bonf_vals = np.minimum(p_raw[upper][valid] * m_tests, 1.0)
    tmp = np.full_like(p_raw[upper], np.nan, dtype=float)
    tmp[valid] = p_bonf_vals
    p_bonf[upper] = tmp
    p_bonf = p_bonf + p_bonf.T  # mirror to lower triangle

# -------------------------------------------------------
# 4) Plotly heatmap with Bonferroni stars + hover details
# -------------------------------------------------------
nice = [label_map.get(v, v) for v in cols]

stars_mat = np.vectorize(p_to_stars)(p_bonf)
text = np.where(
    np.isfinite(rho),
    np.char.add(np.vectorize(lambda x: f"{x:.2f}")(rho), stars_mat),
    ""
)

custom = np.dstack([n_pair, p_raw, p_bonf])  # n, p_raw, p_bonf

fig = go.Figure(
    data=go.Heatmap(
        z=rho,
        x=nice,
        y=nice,
        zmin=-1, zmax=1, zmid=0,
        colorscale="RdBu",
        colorbar=dict(title=("Spearman ρ" if method == "spearman" else "Pearson r")),
        text=text,
        texttemplate="%{text}",
        textfont=dict(size=10),
        customdata=custom,
        hovertemplate=(
            "%{y} vs %{x}"
            "<br>corr=%{z:.2f}"
            "<br>n=%{customdata[0]}"
            "<br>p_raw=%{customdata[1]:.3g}"
            "<br>p_bonf=%{customdata[2]:.3g}"
            f"<br>m_tests={m_tests}"
            "<extra></extra>"
        ),
        hoverongaps=False,
    )
)

fig.update_layout(
    title=f"Baseline correlation matrix of ER features (assess=0, {method}; Bonferroni over m_tests={m_tests})",
    width=950,
    height=950,
)
fig.update_xaxes(tickangle=45)
fig.update_yaxes(autorange="reversed")
fig.show()


In [ ]:
import plotly.figure_factory as ff

# -------------------------
# 0) Lesbare Labels
# -------------------------
label_map = {
    "er_between_sd_mean": "Between-strategy variability",
    "er_within_sd_mean": "Within-strategy variability",
    "er_n_strategies_mean": "Number of strategies used",
    "er_mean_overall": "Overall ER intensity",
    "er_acceptance_mean": "Acceptance",
    "er_control_mean": "Situation controllability",
    "er_distraction_mean": "Distraction",
    "er_intensity_mean": "Negative affect intensity",
    "er_reappraisal_mean": "Reappraisal",
    "er_relaxation_mean": "Relaxation",
    "er_rumination_mean": "Rumination",
    "er_suppression_mean": "Suppression",
    # falls du später SDs dazunimmst:
    "er_acceptance_sd": "Acceptance (SD)",
    "er_control_sd": "Control (SD)",
    "er_distraction_sd": "Distraction (SD)",
    "er_intensity_sd": "Intensity (SD)",
    "er_reappraisal_sd": "Reappraisal (SD)",
    "er_relaxation_sd": "Relaxation (SD)",
    "er_rumination_sd": "Rumination (SD)",
    "er_suppression_sd": "Suppression (SD)",
}

# -------------------------
# 1) Baseline-Daten (assess = 0)
# -------------------------
baseline = df_er_final[df_er_final["assess"] == 0].copy()

# Panel-Aufteilung:
metrics_vars = [
    "er_between_sd_mean",
    "er_within_sd_mean",
]

strategy_vars = [
    "er_acceptance_mean",
    "er_reappraisal_mean",
    "er_relaxation_mean",
    "er_distraction_mean",
    "er_rumination_mean",
    "er_suppression_mean",
    "er_mean_overall",
    "er_intensity_mean",
    "er_control_mean",
]

# Sets für Farbcodierung im Strategien-Plot
adaptive_vars = {
    "er_acceptance_mean",
    "er_reappraisal_mean",
    "er_relaxation_mean",
    "er_distraction_mean",
}

maladaptive_vars = {
    "er_rumination_mean",
    "er_suppression_mean",
}

meta_vars = {
    "er_mean_overall",
    "er_intensity_mean",
    "er_control_mean",
}

blue_palette = ["#1f77b4", "#2ca02c", "#17becf", "#9467bd"]  # adaptive
red_palette = ["#d62728", "#ff7f0e"]                         # maladaptive
meta_palette = ["#7f7f7f", "#525252", "#969696"]             # Meta-Indizes

# -------------------------
# 2) Plot 1: Between-/Within-Variability (eigene Figure)
# -------------------------
hist_data_metrics = []
labels_metrics = []

for v in metrics_vars:
    if v not in baseline.columns:
        continue
    x = baseline[v].dropna()
    if x.empty:
        continue
    hist_data_metrics.append(x.values)
    labels_metrics.append(label_map.get(v, v))

# Farbschema für die beiden Variabilitätsmaße (nicht grau)
metrics_palette = ["#1f77b4", "#ff7f0e"]  # blau, orange
colors_metrics = [metrics_palette[i % len(metrics_palette)] for i in range(len(labels_metrics))]

fig_metrics = ff.create_distplot(
    hist_data_metrics,
    labels_metrics,
    show_hist=False,
    show_rug=False,
    colors=colors_metrics
)

fig_metrics.update_layout(
    title="Baseline distributions of between/within-strategy variability (assess = 0)",
    xaxis_title="Raw value",
    yaxis_title="Density",
    width=800,
    height=400,
)

fig_metrics.show()

# -------------------------
# 3) Plot 2: Strategien + Intensitäten (eigene Figure)
# -------------------------
hist_data_strat = []
labels_strat = []
var_names_strat = []

for v in strategy_vars:
    if v not in baseline.columns:
        continue
    x = baseline[v].dropna()
    if x.empty:
        continue
    hist_data_strat.append(x.values)
    labels_strat.append(label_map.get(v, v))
    var_names_strat.append(v)

colors_strat = []
adapt_idx = 0
malad_idx = 0
meta_idx = 0

for v in var_names_strat:
    if v in adaptive_vars:
        colors_strat.append(blue_palette[adapt_idx % len(blue_palette)])
        adapt_idx += 1
    elif v in maladaptive_vars:
        colors_strat.append(red_palette[malad_idx % len(red_palette)])
        malad_idx += 1
    elif v in meta_vars:
        colors_strat.append(meta_palette[meta_idx % len(meta_palette)])
        meta_idx += 1
    else:
        # Fallback
        colors_strat.append("#7f7f7f")

fig_strat = ff.create_distplot(
    hist_data_strat,
    labels_strat,
    show_hist=False,
    show_rug=False,
    colors=colors_strat
)

fig_strat.update_layout(
    title="Baseline distributions of ER strategies & intensities (assess = 0)",
    xaxis_title="Raw value",
    yaxis_title="Density",
    width=900,
    height=500,
)

fig_strat.show()


### Physical Health

In [ ]:
df_ema_ph = df_ema_content[df_ema_content["quest_title"] == "physical_health"]

extra_cols = ["assess", "study", "quest_create", "quest_nr"]

aggregated_info = df_ema_ph.groupby(["customer", "unique_day_id"])[extra_cols].first().reset_index()

# Pivot the table as specified
df_ph_piv = df_ema_ph.pivot_table(
    index=["customer", "unique_day_id", "assess"],
    columns="quest_title",
    values="choice_text",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_ph_piv.columns = [col for col in df_ph_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_ph_piv = df_ph_piv.reset_index()
df_ph_piv = df_ph_piv.drop_duplicates()

In [ ]:
# make sure they are numeric
df_ph_piv["physical_health"] = df_ph_piv["physical_health"].apply(
    pd.to_numeric, errors="coerce"
)

In [ ]:
agg_ph = (
    df_ph_piv
    .groupby(['customer', 'assess'])["physical_health"]
    .agg(['mean', 'std'])
)

agg_ph = agg_ph.reset_index()
agg_ph.rename({"mean":"ph_mean","std":"ph_std"}, axis=1, inplace=True)

In [ ]:
df_affect_final_filtered = df_affect_final_filtered.merge(agg_ph, on=["customer", "assess"], how="left")

### Social Contact Valence

In [ ]:
df_ema_social = df_ema_content[df_ema_content["quest_title"] == "event_social3"]

extra_cols = ["assess", "study", "quest_create", "quest_nr"]

aggregated_info = df_ema_social.groupby(["customer", "unique_day_id"])[extra_cols].first().reset_index()

# Pivot the table as specified
df_social_piv = df_ema_social.pivot_table(
    index=["customer", "unique_day_id", "assess"],
    columns="quest_title",
    values="choice_text",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_social_piv.columns = [col for col in df_social_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_social_piv = df_social_piv.reset_index()
df_social_piv = df_social_piv.drop_duplicates()

In [ ]:
df_social_piv.groupby("event_social3")["customer"].count()

In [ ]:
# make sure they are numeric
df_social_piv["event_social3"] = df_social_piv["event_social3"].apply(
    pd.to_numeric, errors="coerce"
)

In [ ]:
agg_social = (
    df_social_piv
    .groupby(['customer', 'assess'])["event_social3"]
    .agg(['mean', 'std'])
)

agg_social = agg_social.reset_index()
agg_social.rename({"mean":"social_valence_mean","std":"social_valence_std"}, axis=1, inplace=True)

In [ ]:
df_affect_final_filtered = df_affect_final_filtered.merge(agg_social, on= ["customer", "assess"], how= "left")